In [2]:
#pip install langchain-experimental tabulate

  Using cached PyYAML-6.0.2-cp312-cp312-win_amd64.whl.metadata (2.1 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
     ---------------------------------------- 0.0/60.8 kB ? eta -:--:--
     -------------------------- ------------- 41.0/60.8 kB 1.9 MB/s eta 0:00:01
     ---------------------------------------- 60.8/60.8 kB 1.1 MB/s eta 0:00:00
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached typing_extensions-4.12.2-py3-none-any.whl.metadata (3.0 kB)
  Using cached multidict-6.1.0-cp312-cp312-win_amd64.whl.metadata (5.1 kB)
     ---------------------------------------- 0.0/71.4 kB ? eta -:--:--
     ---------------------------------- ----- 61.4/71.4 kB 1.1 MB/s eta 0:00:01
     ---------------------------------------- 71.4/71.4 kB 1.3 MB/s eta 0:00:00
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached jsonpointer-3.0.0-py2


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_core.tools import tool

In [35]:
import aiohttp
from typing import Dict
import pandas as pd

current_df = pd.read_csv("../assets/orders_example.csv")

#@tool
async def agenerate_forecast(self, target_column: str, prediction_length: int = 30) -> Dict:
        """Genera predicciones usando el endpoint de Chronos"""
        if current_df is None:
            return {"error": "No hay datos cargados"}
            
        # Preparar el archivo CSV para enviarlo
        csv_data = current_df.to_csv(index=False)
        
        # Preparar la solicitud al endpoint
        files = {
            'file': ('../assets/orders_example.csv', csv_data, 'text/csv')
        }
        data = {
            'prediction_length': str(prediction_length),
            'target_column': target_column
        }
        
        async with aiohttp.ClientSession() as session:
            async with session.post('http://localhost:8000/forecast', data=data, file=files) as response:
                if response.status == 200:
                    forecast_results = await response.json()
                    return forecast_results
                else:
                    error_text = await response.text()
                    return {"error": f"Error en la predicción: {error_text}"}
                
#@tool
def generate_forecast(self, target_column: str, prediction_length: int = 30) -> Dict:
        """Genera predicciones usando el endpoint de Chronos"""
        if current_df is None:
            return {"error": "No hay datos cargados"}
            
        # Preparar el archivo CSV para enviarlo
        csv_data = current_df.to_csv(index=False)
        
        # Preparar la solicitud al endpoint
        files = {
            'file': ('../assets/orders_example.csv', csv_data, 'text/csv')
        }
        data = {
            'prediction_length': str(prediction_length),
            'target_column': target_column
        }
        
        with aiohttp.ClientSession() as session:
            with session.post('http://localhost:8000/forecast', data=data, files=files) as response:
                if response.status == 200:
                    forecast_results = response.json()
                    return forecast_results
                else:
                    error_text = response.text()
                    return {"error": f"Error en la predicción: {error_text}"}

In [24]:
print(generate_forecast.name)
print(generate_forecast.description)
print(generate_forecast.args)

generate_forecast
Genera predicciones usando el endpoint de Chronos
{'self': {'title': 'Self'}, 'target_column': {'title': 'Target Column', 'type': 'string'}, 'prediction_length': {'default': 30, 'title': 'Prediction Length', 'type': 'integer'}}


In [28]:
from langchain_core.tools import StructuredTool
from typing_extensions import Annotated

forecasterTool = StructuredTool.from_function(func=generate_forecast, coroutine=agenerate_forecast)

print( await forecasterTool.ainvoke(target_column="orders", prediction_length=30))

AttributeError: 'StructuredTool' object has no attribute '__name__'

In [36]:
result = await agenerate_forecast("orders", 30)
print(result)

TypeError: ClientSession._request() got an unexpected keyword argument 'file'

In [41]:
import requests
async def generate_forecast_request(target_column: str, prediction_length: int = 30) -> Dict:
    """Genera predicciones usando el endpoint de Chronos"""
    if current_df is None:
        return {"error": "No hay datos cargados"}

    # Preparar el archivo CSV para enviarlo
    csv_data = current_df.to_csv(index=False)

    # Preparar la solicitud al endpoint
    files = {
        'file': ('../assets/orders_example.csv', csv_data, 'text/csv')
    }
    data = {
        'prediction_length': str(prediction_length),
        'target_column': target_column
    }

    response = requests.post('http://localhost:8000/forecast', data=data, files=files)
    if response.status_code == 200:
        forecast_results = response.json()
        return forecast_results
    else:
        error_text = response.text
        return {"error": f"Error en la predicción: {error_text}"}
    
print(await generate_forecast_request("orders", 30))

{'actual': [152, 485, 398, 320, 156, 121, 238, 70, 152, 171, 264, 380, 137, 422, 149, 409, 201, 180, 199, 358, 307, 393, 463, 343, 435, 241, 493, 326, 210, 363, 71, 302, 285, 394, 98, 108, 219, 237, 320, 239, 495, 224, 495, 100, 413, 104, 293, 369], 'forecast': [311.3306884765625, 316.21722412109375, 311.8193359375, 289.34130859375, 267.5962829589844, 255.13563537597656, 245.85122680664062, 242.4306640625, 245.60690307617188, 248.41665649414062, 248.0501708984375, 252.8145294189453, 259.53350830078125, 272.1773986816406, 277.94195556640625, 284.76019287109375, 291.90673828125, 292.45648193359375, 298.5035400390625, 301.8019714355469, 308.3987731933594, 312.06365966796875, 317.4388427734375, 317.927490234375, 316.4615478515625, 320.6150817871094, 315.4842224121094, 295.9381103515625, 273.7349853515625, 259.6556701660156], 'horizon': [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]}


In [9]:
dict_nulo = {"value1": None, "value2": None, "value3": "Hola"}
import json

dict_nulo_string = json.dumps(dict_nulo, ensure_ascii=False)    
print(dict_nulo_string)

dict_nulo_string = json.loads(dict_nulo_string)
print(dict_nulo_string)


{"value1": null, "value2": null, "value3": "Hola"}
{'value1': None, 'value2': None, 'value3': 'Hola'}


In [4]:
type(dict_nulo_string)

dict

In [5]:
try:
    json.dumps(dict_nulo_string)
    print("dict_nulo_string is serializable")
except (TypeError, OverflowError) as e:
    print(f"dict_nulo_string is not serializable: {e}")

dict_nulo_string is serializable


In [10]:
json_long_string = "{\"text\": \"La tendencia principal es que los valores pronosticados est\u00e1n ligeramente por debajo de los valores reales, con una diferencia promedio de alrededor de 7.5. Adem\u00e1s, la desviaci\u00f3n est\u00e1ndar de los valores pronosticados es significativamente menor que la de los valores reales. Esto sugiere que el modelo de pron\u00f3stico puede no estar capturando completamente la variabilidad de los datos reales. Las implicaciones para el negocio son que se debe revisar y ajustar el modelo de pron\u00f3stico para mejorar la precisi\u00f3n de las predicciones y tomar decisiones informadas basadas en los datos reales en lugar de en las predicciones.\", \"facialExpression\": \"default\", \"animation\": \"Talking_0\", \"type\": \"data_analytics\", \"chartData\": [{\"date\": \"2021-01\", \"variables\": {\"orders\": 152.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-02\", \"variables\": {\"orders\": 485.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-03\", \"variables\": {\"orders\": 398.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-04\", \"variables\": {\"orders\": 320.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-05\", \"variables\": {\"orders\": 156.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-06\", \"variables\": {\"orders\": 121.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-07\", \"variables\": {\"orders\": 238.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-08\", \"variables\": {\"orders\": 70.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-09\", \"variables\": {\"orders\": 152.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-10\", \"variables\": {\"orders\": 171.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-11\", \"variables\": {\"orders\": 264.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2021-12\", \"variables\": {\"orders\": 380.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-01\", \"variables\": {\"orders\": 137.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-02\", \"variables\": {\"orders\": 422.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-03\", \"variables\": {\"orders\": 149.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-04\", \"variables\": {\"orders\": 409.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-05\", \"variables\": {\"orders\": 201.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-06\", \"variables\": {\"orders\": 180.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-07\", \"variables\": {\"orders\": 199.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-08\", \"variables\": {\"orders\": 358.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-09\", \"variables\": {\"orders\": 307.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-10\", \"variables\": {\"orders\": 393.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-11\", \"variables\": {\"orders\": 463.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2022-12\", \"variables\": {\"orders\": 343.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-01\", \"variables\": {\"orders\": 435.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-02\", \"variables\": {\"orders\": 241.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-03\", \"variables\": {\"orders\": 493.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-04\", \"variables\": {\"orders\": 326.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-05\", \"variables\": {\"orders\": 210.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-06\", \"variables\": {\"orders\": 363.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-07\", \"variables\": {\"orders\": 71.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-08\", \"variables\": {\"orders\": 302.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-09\", \"variables\": {\"orders\": 285.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-10\", \"variables\": {\"orders\": 394.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-11\", \"variables\": {\"orders\": 98.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2023-12\", \"variables\": {\"orders\": 108.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-01\", \"variables\": {\"orders\": 219.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-02\", \"variables\": {\"orders\": 237.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-03\", \"variables\": {\"orders\": 320.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-04\", \"variables\": {\"orders\": 239.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-05\", \"variables\": {\"orders\": 495.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-06\", \"variables\": {\"orders\": 224.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-07\", \"variables\": {\"orders\": 495.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-08\", \"variables\": {\"orders\": 100.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-09\", \"variables\": {\"orders\": 413.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-10\", \"variables\": {\"orders\": 104.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-11\", \"variables\": {\"orders\": 293.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2024-12\", \"variables\": {\"orders\": 369.0, \"lower_quantile\": null, \"upper_quantile\": null}}, {\"date\": \"2025-01-01 00:00:00\", \"variables\": {\"orders\": 311.3306884765625, \"lower_quantile\": 145.1887969970703, \"upper_quantile\": 502.88250732421875}}, {\"date\": \"2025-02-01 00:00:00\", \"variables\": {\"orders\": 316.21722412109375, \"lower_quantile\": 135.41574096679688, \"upper_quantile\": 527.3151245117188}}, {\"date\": \"2025-03-01 00:00:00\", \"variables\": {\"orders\": 311.8193359375, \"lower_quantile\": 127.59730529785156, \"upper_quantile\": 533.178955078125}}, {\"date\": \"2025-04-01 00:00:00\", \"variables\": {\"orders\": 289.34130859375, \"lower_quantile\": 108.05119323730469, \"upper_quantile\": 514.6101684570312}}, {\"date\": \"2025-05-01 00:00:00\", \"variables\": {\"orders\": 267.5962829589844, \"lower_quantile\": 89.48239135742188, \"upper_quantile\": 494.0867614746094}}, {\"date\": \"2025-06-01 00:00:00\", \"variables\": {\"orders\": 255.13563537597656, \"lower_quantile\": 80.6866455078125, \"upper_quantile\": 484.313720703125}}, {\"date\": \"2025-07-01 00:00:00\", \"variables\": {\"orders\": 245.85122680664062, \"lower_quantile\": 75.80012512207031, \"upper_quantile\": 476.4952697753906}}, {\"date\": \"2025-08-01 00:00:00\", \"variables\": {\"orders\": 242.4306640625, \"lower_quantile\": 73.84550476074219, \"upper_quantile\": 475.5179443359375}}, {\"date\": \"2025-09-01 00:00:00\", \"variables\": {\"orders\": 245.60690307617188, \"lower_quantile\": 75.80012512207031, \"upper_quantile\": 482.3591003417969}}, {\"date\": \"2025-10-01 00:00:00\", \"variables\": {\"orders\": 248.41665649414062, \"lower_quantile\": 76.77742004394531, \"upper_quantile\": 486.268310546875}}, {\"date\": \"2025-11-01 00:00:00\", \"variables\": {\"orders\": 248.0501708984375, \"lower_quantile\": 76.77742004394531, \"upper_quantile\": 482.3591003417969}}, {\"date\": \"2025-12-01 00:00:00\", \"variables\": {\"orders\": 252.8145294189453, \"lower_quantile\": 83.61856079101562, \"upper_quantile\": 485.291015625}}]}" 
json_long_string = json.loads(json_long_string)

In [11]:
json_long_string

{'text': 'La tendencia principal es que los valores pronosticados están ligeramente por debajo de los valores reales, con una diferencia promedio de alrededor de 7.5. Además, la desviación estándar de los valores pronosticados es significativamente menor que la de los valores reales. Esto sugiere que el modelo de pronóstico puede no estar capturando completamente la variabilidad de los datos reales. Las implicaciones para el negocio son que se debe revisar y ajustar el modelo de pronóstico para mejorar la precisión de las predicciones y tomar decisiones informadas basadas en los datos reales en lugar de en las predicciones.',
 'facialExpression': 'default',
 'animation': 'Talking_0',
 'type': 'data_analytics',
 'chartData': [{'date': '2021-01',
   'variables': {'orders': 152.0,
    'lower_quantile': None,
    'upper_quantile': None}},
  {'date': '2021-02',
   'variables': {'orders': 485.0,
    'lower_quantile': None,
    'upper_quantile': None}},
  {'date': '2021-03',
   'variables': {